## HumanEvals

We repeat the same thing here, but now instead of comparing the nll of the target and the predicted tokens, we take the probabilities of the predicted token and then sum their logs together. 

In [2]:
!pip install -qU torch  
!pip install -qU transformers
!pip install -qU datasets altair huggingface_hub tqdm accelerate python-dotenv
!pip install -qU sentencepiece

import torch as t
from torch.nn import functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from datasets import load_dataset
import altair as alt
from huggingface_hub import notebook_login
from tqdm import tqdm

import os
from dotenv import load_dotenv

device = "cuda" if t.cuda.is_available() else "cpu"


[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 24.0
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
stdout = sys.stdout

In [4]:
!nvidia-smi

Wed Jun  5 03:37:43 2024       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          Off |   00000000:00:0C.0 Off |                    0 |
| N/A   32C    P0             54W /  300W |       3MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
HF_TOKEN = os.getenv("HF_TOKEN")
notebook_login()

In [6]:
GPT2_MODEL = "openai-community/gpt2-medium"
MISTRAL_MODEL = "mistralai/Mistral-7B-v0.3"
LLAMA3_MODEL = "meta-llama/Meta-Llama-3-8B"
LLAMA2_MODEL = "meta-llama/Llama-2-7b-hf"
GEMMA_MODEL = "google/gemma-7b"
OPENCHAT_MODEL = "openchat/openchat-3.6-8b-20240522"

## Dataset

We're using the original humaneval dataset (comment -> function). 

In [9]:
dataset = load_dataset("THUDM/humaneval-x", split="test")
prompts = dataset["prompt"]

## Calculating the Perplexity

The function to calculate the perplexity is slightly different because of the change in how I calculate the problabilities. 

In [11]:
from transformers.generation.logits_process import TopKLogitsWarper, TopPLogitsWarper

def top_k_top_p_filtering(
    logits: t.FloatTensor,
    top_k: int = 0,
    top_p: float = 1.0,
    filter_value: float = -float("Inf"),
    min_tokens_to_keep: int = 1,
) -> t.FloatTensor:

    if top_k > 0:
        logits = TopKLogitsWarper(top_k=top_k, filter_value=filter_value, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )

    if 0 <= top_p <= 1.0:
        logits = TopPLogitsWarper(top_p=top_p, filter_value=filter_value, min_tokens_to_keep=min_tokens_to_keep)(
            None, logits
        )

    return logits

def calculate_mean_nll_loss(model, tokenizer, prompt):

    num_sampled = 0
    nll_vals = []    
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    while True:
        outputs = model(input_ids)
        # Get logits from last layer
        last_layer_logits = outputs.logits[:, -1, :]

        top_logits = top_k_top_p_filtering(last_layer_logits, top_k=10, top_p=1.0)
        probs = F.softmax(top_logits, dim=-1)
        token_idx = t.argmax(probs, dim=-1)
        probability = probs[0,token_idx]
        generated_next_token = t.multinomial(probs, num_samples=1)
        nll_vals.append(-1 * t.log(probability))

        # Once the model is done predicting, we calculate the perplexity by taking the 
        # exp of the mean nll values.
        if generated_next_token == tokenizer.eos_token_id:
            print("Ended")
            ppl = t.exp(t.stack(nll_vals, dim=0).mean())
            return ppl
        
        input_ids = t.cat([input_ids, generated_next_token], dim=-1)
        num_sampled += 1

In [12]:
model = AutoModelForCausalLM.from_pretrained(LLAMA3_MODEL).to(device)
tokenizer = AutoTokenizer.from_pretrained(LLAMA3_MODEL)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [13]:
output = calculate_mean_nll_loss(model, tokenizer, prompts[0])

/tmp/ipykernel_121629/1670087896.py:34: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  probs = F.softmax(top_logits)


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 